<h1>PyArrow Functionality</h1>

<p>pandas can utilize <a href="https://arrow.apache.org/docs/python/index.html">PyArrow</a> to extend functionality and improve the performance of various APIs. This includes:</p>

<ul class="simple">
<li><p>More extensive <a href="https://arrow.apache.org/docs/python/api/datatypes.html">data types</a> compared to NumPy</p></li>
<li><p>Missing data support (NA) for all data types</p></li>
<li><p>Performant IO reader integration</p></li>
<li><p>Facilitate interoperability with other dataframe libraries based on the Apache Arrow specification (e.g. polars, cuDF)</p></li>
</ul>

<p>To use this functionality, please ensure you have
<a href="https://pandas.pydata.org/docs/getting_started/install.html#install-optional-dependencies">installed the minimum supported PyArrow version.</a></p>

# <h2>Data Structure Integration</h2>

<p>A <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.html" title="pandas.Series"><code>Series</code></a>,
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html" title="pandas.Index"><code>Index</code></a>, or the columns of a
<a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html" title="pandas.DataFrame"><code>DataFrame</code></a> can be directly backed by a
<a href="https://arrow.apache.org/docs/python/generated/pyarrow.ChunkedArray.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.ChunkedArray</code></a> which is similar to a NumPy array.
To construct these from the main pandas data structures, you can pass in a string of the type followed by <code>[pyarrow]</code>, e.g. <code>"int64[pyarrow]""</code> into the <code>dtype</code> parameter</p>

In [5]:
import pandas as pd

In [6]:
ser = pd.Series([-1.5, 0.2, None], dtype='float32[pyarrow]')

In [7]:
ser

,0
0,-1.5
1,0.2
2,<NA>


In [8]:
idx = pd.Index([True, None], dtype='bool[pyarrow]')

In [9]:
idx

Index([True, <NA>], dtype='bool[pyarrow]')

In [10]:
df = pd.DataFrame([[1, 2], [3, 4]], dtype='uint64[pyarrow]')

In [11]:
df

,0,1
0,1,2
1,3,4


<div class="alert alert-block alert-info">
<p>Note</p>
<p>The string alias <code>"string[pyarrow]"</code> maps to <code>pd.StringDtype("pyarrow")</code> which is not equivalent to specifying <code>dtype=pd.ArrowDtype(pa.string())</code>. Generally, operations on the data will behave similarly except <code>pd.StringDtype("pyarrow")</code> can return NumPy-backed nullable types while <code>pd.ArrowDtype(pa.string())</code> will return
<a href="https://pandas.pydata.org/docs/reference/api/pandas.ArrowDtype.html" title="pandas.ArrowDtype"><code>ArrowDtype</code></a>.</p>

In [12]:
import pyarrow as pa

In [13]:
data = list("abc")

In [14]:
ser_sd = pd.Series(data, dtype='string[pyarrow]')

In [15]:
ser_ad = pd.Series(data, dtype=pd.ArrowDtype(pa.string()))

In [16]:
ser_ad.dtype == ser_sd.dtype

False

In [17]:
ser_sd.str.contains("a")

,0
0,True
1,False
2,False


In [18]:
ser_ad.str.contains("a")

,0
0,True
1,False
2,False


<p>For PyArrow types that accept parameters, you can pass in a PyArrow type with those parameters into
<a href="https://pandas.pydata.org/docs/reference/api/pandas.ArrowDtype.html" title="pandas.ArrowDtype"><code>ArrowDtype</code></a> to use in the <code>dtype</code> parameter.</p>

In [19]:
import pyarrow as pa

In [20]:
list_str_type = pa.list_(pa.string())

In [21]:
ser = pd.Series([["hello"], ["there"]], dtype=pd.ArrowDtype(list_str_type))

In [22]:
ser

,0
0,['hello']
1,['there']


In [23]:
from datetime import time

In [24]:
idx = pd.Index([time(12, 30), None], dtype=pd.ArrowDtype(pa.time64("us")))

In [25]:
idx

Index([12:30:00, <NA>], dtype='time64[us][pyarrow]')

In [26]:
from decimal import Decimal

In [27]:
decimal_type = pd.ArrowDtype(pa.decimal128(3, scale=2))

In [28]:
data = [[Decimal("3.19"), None], [None, Decimal("-1.23")]]

In [29]:
df = pd.DataFrame(data, dtype=decimal_type)

In [30]:
df

,0,1
0,3.19,<NA>
1,<NA>,-1.23


<p>If you already have an <a href="https://arrow.apache.org/docs/python/generated/pyarrow.Array.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.Array</code></a> or <a href="https://arrow.apache.org/docs/python/generated/pyarrow.ChunkedArray.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.ChunkedArray</code></a>, you can pass it into
<a href="https://pandas.pydata.org/docs/reference/api/pandas.arrays.ArrowExtensionArray.html" title="pandas.arrays.ArrowExtensionArray"><code>arrays.ArrowExtensionArray</code></a> to construct the associated <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.html" title="pandas.Series"><code>Series</code></a>, <a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html" title="pandas.Index"><code>Index</code></a> or
<a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html" title="pandas.DataFrame"><code>DataFrame</span></code></a> object.</p>

In [31]:
pa_array = pa.array(
    [{"1": "2"}, {"10": "20"}, None],
    type=pa.map_(pa.string(), pa.string()),
)

In [32]:
ser = pd.Series(pd.arrays.ArrowExtensionArray(pa_array))

In [33]:
ser

,0
0,"[('1', '2')]"
1,"[('10', '20')]"
2,<NA>


<p>To retrieve a pyarrow <a href="https://arrow.apache.org/docs/python/generated/pyarrow.ChunkedArray.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.ChunkedArray</code></a> from a
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.html" title="pandas.Series"><code>Series</code></a> or <a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html" title="pandas.Index"><code>Index</code></a>, you can call the pyarrow array constructor on the
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.html" title="pandas.Series"><code>Series</code></a> or <a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html" title="pandas.Index"><code>Index</code></a>.</p>

In [34]:
ser = pd.Series([1, 2, None], dtype='uint8[pyarrow]')

In [35]:
pa.array(ser)

[
  1,
  2,
  null
]

In [36]:
idx = pd.Index(ser)

In [37]:
pa.array(idx)

[
  1,
  2,
  null
]

<p>To convert a <a href="https://arrow.apache.org/docs/python/generated/pyarrow.Table.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.Table</code></a> to a <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html" title="pandas.DataFrame"><code>DataFrame</code></a>, you can call the
<a href="https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_pandas" title="(in Apache Arrow v21.0.0)"><code>pyarrow.Table.to_pandas()</code></a> method with <code>types_mapper=pd.ArrowDtype</code>.</p>

In [38]:
table = pa.table([pa.array([1, 2, 3], type=pa.int64())], names=["a"])

In [39]:
df = table.to_pandas(types_mapper=pd.ArrowDtype)

In [40]:
df

,a
0,1
1,2
2,3


In [41]:
df.dtypes

,0
a,int64[pyarrow]


# <h2>Operations</h2>

<p>PyArrow data structure integration is implemented through pandas’
<a href="https://pandas.pydata.org/docs/reference/api/pandas.api.extensions.ExtensionArray.html" title="pandas.api.extensions.ExtensionArray"><code>ExtensionArray</code></a> <a href="https://pandas.pydata.org/docs/development/extending.html#extending-extension-type">interface</a>; therefore, supported functionality exists where this interface is integrated within the pandas API. Additionally, this functionality is accelerated with PyArrow <a href="https://arrow.apache.org/docs/python/api/compute.html">compute functions</a> where available. This includes:</p>

<ul class="simple">
<li><p>Numeric aggregations</p></li>
<li><p>Numeric arithmetic</p></li>
<li><p>Numeric rounding</p></li>
<li><p>Logical and comparison functions</p></li>
<li><p>String functionality</p></li>
<li><p>Datetime functionality</p></li>
</ul>

<p>The following are just some examples of operations that are accelerated by native PyArrow compute functions.</p>

In [42]:
import pyarrow as pa

In [43]:
ser = pd.Series([-1.545, 0.211, None], dtype='float32[pyarrow]')

In [44]:
ser.mean()

-0.6669999808073044

In [45]:
ser + ser

,0
0,-3.09
1,0.422
2,<NA>


In [46]:
ser > (ser + 1)

,0
0,False
1,False
2,<NA>


In [47]:
ser.dropna()

,0
0,-1.545
1,0.211


In [48]:
ser.isna()

,0
0,False
1,False
2,True


In [59]:
ser.fillna(0)

,0
0,-1.545
1,0.211
2,0.0


In [61]:
ser_str = pd.Series(["a", "b", None], dtype=pd.ArrowDtype(pa.string()))

In [62]:
ser_str.str.startswith("a")

,0
0,True
1,False
2,<NA>


In [63]:
from datetime import datetime

In [64]:
pa_type = pd.ArrowDtype(pa.timestamp('ns'))

In [65]:
ser_dt = pd.Series([datetime(2022, 1, 1), None], dtype=pa_type)

In [66]:
ser_dt.dt.strftime('%Y-%m')

,0
0,2022-01
1,<NA>


# <h2>I/O Reading</h2>

<p>PyArrow also provides IO reading functionality that has been integrated into several pandas IO readers. The following functions provide an <code>engine</code> keyword that can dispatch to PyArrow to accelerate reading from an IO source.</p>

<ul class="simple">
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html" title="pandas.read_csv"><code>read_csv()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.read_json.html" title="pandas.read_json"><code>read_json()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.read_orc.html" title="pandas.read_orc"><code>read_orc()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.read_feather.html" title="pandas.read_feather"><code>read_feather()</code></a></p></li>
</ul>

In [49]:
import io

In [50]:
data = io.StringIO("""a,b,c
   1,2.5,True
   3,4.5,False
""")

In [51]:
df = pd.read_csv(data, engine='pyarrow')

In [52]:
df

,a,b,c
0,1,2.5,True
1,3,4.5,False


<p>By default, these functions and all other IO reader functions return NumPy-backed data. These readers can return PyArrow-backed data by specifying the parameter <code>dtype_backend="pyarrow"</code>.
A reader does not need to set <code>engine="pyarrow"</code> to necessarily return PyArrow-backed data.</p>

In [53]:
import io

In [54]:
data = io.StringIO("""a,b,c,d,e,f,g,h,i
    1,2.5,True,a,,,,,
    3,4.5,False,b,6,7.5,True,a,
""")

In [55]:
df_pyarrow = pd.read_csv(data, dtype_backend="pyarrow")

In [56]:
df_pyarrow.dtypes

,0
a,int64[pyarrow]
b,double[pyarrow]
c,bool[pyarrow]
d,string[pyarrow]
e,int64[pyarrow]
f,double[pyarrow]
g,bool[pyarrow]
h,string[pyarrow]
i,null[pyarrow]


<p>Several non-IO reader functions can also use the <code>dtype_backend</code> argument to return PyArrow-backed data including:</p>

<ul class="simple">
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html" title="pandas.to_numeric"><code>to_numeric()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.convert_dtypes.html" title="pandas.DataFrame.convert_dtypes"><code>DataFrame.convert_dtypes()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.convert_dtypes.html" title="pandas.Series.convert_dtypes"><code>Series.convert_dtypes()</code></a></p></li>
</ul>